# 첫 스크립트와 디버거

같은 가격과 개수로 잘못된 덧셈 결과를 디버거에서 확인하고 곱셈으로 수정합니다.

## 실행 환경 준비

개인 파일을 바꾸지 않도록 임시 폴더에 Python 스크립트를 작성합니다.

In [1]:
import subprocess
import sys
from pathlib import Path
from tempfile import TemporaryDirectory

# 임시 폴더는 노트북 실행이 끝나면 삭제됩니다.
temporary_folder = TemporaryDirectory()
workspace = Path(temporary_folder.name)
bug_file = workspace / "price_bug.py"

## 기본 실행: 정상 총액 계산

가격과 개수를 곱해 6,000원이 나오는 기본 코드를 먼저 확인합니다.

In [2]:
# price는 한 개의 가격, count는 상품 개수입니다.
price = 3000
count = 2
total = price * count
print(f"정상 총액: {total}원")
assert total == 6000

정상 총액: 6000원


## Before: 덧셈으로 작성한 스크립트

가격 3,000원과 개수 2개를 `+`로 계산하면 코드는 실행되지만 총액은 틀립니다.

In [3]:
# write_text는 문자열을 Python 파일로 저장하고 쓴 문자 수를 반환합니다.
bug_source = """price = 3000
count = 2
total = price + count
print(total)
"""
bug_file.write_text(bug_source, encoding="utf-8")
bug_run = subprocess.run([sys.executable, bug_file], capture_output=True, text=True, check=True)
bug_total = int(bug_run.stdout.strip())
print(f"잘못된 총액: {bug_total}원")
assert bug_total == 3002

잘못된 총액: 3002원


## 디버거로 중간 값 확인

`pdb`에서 4번 줄 직전에 멈춘 뒤 `price`, `count`, `total`을 직접 출력합니다.

In [4]:
# pdb에 중단점, 계속 실행, 변수 출력, 종료 명령을 순서대로 전달합니다.
debug_commands = """b 4
c
p price
p count
p total
q
"""
debug_run = subprocess.run(
    [sys.executable, "-m", "pdb", bug_file.name],
    cwd=workspace, input=debug_commands, capture_output=True, text=True, check=True,
)
# pdb 프롬프트 뒤의 숫자 결과만 선택하여 개인 경로를 제외합니다.
observed = [line.removeprefix("(Pdb) ") for line in debug_run.stdout.splitlines()
            if line.removeprefix("(Pdb) ").isdigit()]
print(f"price = {observed[-3]}")
print(f"count = {observed[-2]}")
print(f"total = {observed[-1]}")
assert observed[-3:] == ["3000", "2", "3002"]

price = 3000
count = 2
total = 3002


## After: 곱셈으로 수정

값은 그대로 두고 연산자만 `+`에서 `*`로 바꿥니다.

In [5]:
# 가격과 개수를 곱하도록 연산자 하나만 수정합니다.
fixed_file = workspace / "hello.py"
fixed_source = bug_source.replace("price + count", "price * count")
fixed_file.write_text(fixed_source, encoding="utf-8")
fixed_run = subprocess.run([sys.executable, fixed_file], capture_output=True, text=True, check=True)
fixed_total = int(fixed_run.stdout.strip())
print(f"수정한 총액: {fixed_total}원")
assert fixed_total == 6000
temporary_folder.cleanup()

수정한 총액: 6000원


## 결과 해석

디버거에서 입력값 3,000과 2가 예상대로였지만 `total`이 3,002였음을 확인했습니다. 연산자를 곱셈으로 바꾼 뒤 같은 입력에서 6,000원을 확인했습니다.